In [ ]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import torchvision.models as models

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

!pip install torchmetrics
from torchmetrics.classification import MulticlassAUROC

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 16.6 MB/s eta 0:00:00


In [ ]:
# ================================
# Dataset Download
# ================================

import kagglehub

path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")

print("Dataset Path:", path)


Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
Dataset Path: /kaggle/input/skin-cancer-mnist-ham10000


In [ ]:
# ================================
# Load Metadata
# ================================

meta_path = os.path.join(path, "HAM10000_metadata.csv")

df = pd.read_csv(meta_path)

print("Total Samples:", len(df))
df.head()


Total Samples: 10015


,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [ ]:
# ================================
# Metadata Preprocessing
# ================================

# Fill missing age with median
df["age"].fillna(df["age"].median(), inplace=True)

# Fill categorical with mode
for col in ["sex", "localization", "dx_type"]:
    df[col].fillna(df[col].mode()[0], inplace=True)

# One-hot encode categorical
df_encoded = pd.get_dummies(
    df,
    columns=["sex", "localization", "dx_type"]
)

# Standardize age
scaler = StandardScaler()
df_encoded["age"] = scaler.fit_transform(df_encoded[["age"]])

print("Encoded Features:", df_encoded.shape)


Encoded Features: (10015, 26)


/tmp/ipython-input-253524042.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["age"].fillna(df["age"].median(), inplace=True)
/tmp/ipython-input-253524042.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try u

In [ ]:
# ================================
# Encode Labels
# ================================

label_map = {
    "akiec":0,
    "bcc":1,
    "bkl":2,
    "df":3,
    "mel":4,
    "nv":5,
    "vasc":6
}

df_encoded["label"] = df_encoded["dx"].map(label_map)

num_classes = 7


In [ ]:
# ================================
# 70/30 Split
# ================================

train_df, test_df = train_test_split(
    df_encoded,
    test_size=0.3,
    stratify=df_encoded["label"],
    random_state=42
)

print("Train:", len(train_df))
print("Test :", len(test_df))


Train: 7010
Test : 3005


In [ ]:
# ================================
# Image Preprocessing
# ================================

train_transform = T.Compose([

    # Resize to 256x256
    T.Resize((256,256)),

    # Augmentation (Section 3.2.2)
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(15),

    T.ToTensor(),

    # Normalize to [0,1]
])

test_transform = T.Compose([
    T.Resize((256,256)),
    T.ToTensor()
])


In [ ]:
# ================================
# Dataset Class
# ================================

class HAMDataset(Dataset):

    def __init__(self, df, img_dir, transform=None):

        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

        self.meta_cols = [c for c in df.columns
                          if c not in ["image_id","dx","label","dx_type", "lesion_id"]]


    def __len__(self):
        return len(self.df)


    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        img_id = row["image_id"] + ".jpg"

        # Images are in two folders
        img_path1 = os.path.join(
            self.img_dir, "HAM10000_images_part_1", img_id
        )

        img_path2 = os.path.join(
            self.img_dir, "HAM10000_images_part_2", img_id
        )

        if os.path.exists(img_path1):
            img_path = img_path1
        else:
            img_path = img_path2

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        # Metadata vector
        meta = row[self.meta_cols].values.astype(np.float32)

        label = row["label"]

        return image, meta, label

In [ ]:
# ================================
# DataLoaders
# ================================

train_ds = HAMDataset(
    train_df,
    path,
    train_transform
)

test_ds = HAMDataset(
    test_df,
    path,
    test_transform
)

train_loader = DataLoader(
    train_ds,
    batch_size=64,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_ds,
    batch_size=64,
    shuffle=False,
    num_workers=2
)

In [ ]:
# ================================
# Image Encoder
# ================================

class ImageEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        base = models.resnet18(pretrained=True)

        self.backbone = nn.Sequential(
            *list(base.children())[:-1]
        )

        self.fc = nn.Linear(512,512)


    def forward(self,x):

        x = self.backbone(x)
        x = x.view(x.size(0), -1)

        x = self.fc(x)

        return x

In [ ]:
# ================================
# Clinical MLP
# ================================

class ClinicalMLP(nn.Module):

    def __init__(self, in_dim):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(in_dim,128),
            nn.ReLU(),

            nn.Linear(128,256),
            nn.ReLU()
        )


    def forward(self,x):

        return self.net(x)

In [ ]:
# ================================
# Cross-Attention Fusion
# ================================

class CrossAttention(nn.Module):

    def __init__(self, dim_q=512, dim_kv=256):

        super().__init__()

        self.q = nn.Linear(dim_q,256)
        self.k = nn.Linear(dim_kv,256)
        self.v = nn.Linear(dim_kv,256)

        self.scale = 256 ** 0.5


    def forward(self, img_feat, meta_feat):

        Q = self.q(img_feat).unsqueeze(1)
        K = self.k(meta_feat).unsqueeze(1)
        V = self.v(meta_feat).unsqueeze(1)

        attn = torch.softmax(
            torch.bmm(Q, K.transpose(1,2)) / self.scale,
            dim=-1
        )

        out = torch.bmm(attn, V)

        return out.squeeze(1)

In [ ]:
# ================================
# Full Model
# ================================

class MultiModalNet(nn.Module):

    def __init__(self, meta_dim):

        super().__init__()

        self.img_enc = ImageEncoder()

        self.meta_enc = ClinicalMLP(meta_dim)

        self.fusion = CrossAttention()

        self.classifier = nn.Sequential(
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Linear(128,7)
        )


    def forward(self, img, meta):

        img_f = self.img_enc(img)

        meta_f = self.meta_enc(meta)

        fused = self.fusion(img_f, meta_f)

        out = self.classifier(fused)

        return out

In [ ]:
# ================================
# Training Setup
# ================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

meta_dim = train_ds[0][1].shape[0]

model = MultiModalNet(meta_dim).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 169MB/s]


In [ ]:
# ================================
# Training Loop
# ================================

def train_epoch(model,loader):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for imgs,meta,labels in loader:

        imgs = imgs.to(device)
        meta = meta.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        out = model(imgs,meta)

        loss = criterion(out,labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        pred = out.argmax(1)

        correct += (pred==labels).sum().item()
        total += labels.size(0)

    acc = correct/total

    return total_loss/len(loader), acc

In [ ]:
# ================================
# Evaluation
# ================================

def evaluate(model,loader):

    model.eval()

    preds = []
    targets = []

    with torch.no_grad():

        for imgs,meta,labels in loader:

            imgs = imgs.to(device)
            meta = meta.to(device)

            out = model(imgs,meta)

            pred = out.argmax(1).cpu()

            preds.extend(pred.numpy())
            targets.extend(labels.numpy())

    return np.array(preds), np.array(targets)


In [ ]:
# ================================
# Training (100 Epochs)
# ================================

EPOCHS = 100

for epoch in range(EPOCHS):

    loss, acc = train_epoch(model, train_loader)

    if (epoch+1) % 5 == 0:

        print(f"Epoch [{epoch+1}/{EPOCHS}] "
              f"Loss: {loss:.4f} "
              f"Acc: {acc:.4f}")


Epoch [5/100] Loss: 0.7137 Acc: 0.7126
Epoch [10/100] Loss: 0.6969 Acc: 0.7201
Epoch [15/100] Loss: 0.6886 Acc: 0.7225
Epoch [20/100] Loss: 0.6733 Acc: 0.7271
Epoch [25/100] Loss: 0.6655 Acc: 0.7261
Epoch [30/100] Loss: 0.6580 Acc: 0.7314
Epoch [35/100] Loss: 0.6556 Acc: 0.7298
Epoch [40/100] Loss: 0.6472 Acc: 0.7344
Epoch [45/100] Loss: 0.6415 Acc: 0.7327
Epoch [50/100] Loss: 0.6327 Acc: 0.7402
Epoch [55/100] Loss: 0.6280 Acc: 0.7375
Epoch [60/100] Loss: 0.6224 Acc: 0.7442
Epoch [65/100] Loss: 0.6187 Acc: 0.7429
Epoch [70/100] Loss: 0.6136 Acc: 0.7448
Epoch [75/100] Loss: 0.6064 Acc: 0.7414
Epoch [80/100] Loss: 0.6057 Acc: 0.7447
Epoch [85/100] Loss: 0.6038 Acc: 0.7475
Epoch [90/100] Loss: 0.5999 Acc: 0.7482
Epoch [95/100] Loss: 0.5995 Acc: 0.7486
Epoch [100/100] Loss: 0.5994 Acc: 0.7476
